# Chassis Impact Detector -- Stage 3 Training (Kaggle)

Trains the lightweight CNN on the log-mel spectrogram windows produced by Stage 2
(`ai_audio/preprocessing/make_features.py`, run on the Pi), then exports the result
to ONNX for Stage 4 (NCNN inference back on the Pi).

**Before running:**
1. On the Pi: `cd ai_audio/dataset && zip -r processed_dataset.zip processed`
2. Upload `processed_dataset.zip` as a new Kaggle Dataset (kaggle.com -> Datasets -> New Dataset).
3. Attach that dataset to this notebook (right sidebar -> Add Input).
4. Turn on a GPU accelerator: Settings -> Accelerator -> GPU T4 x2 (or P100).
5. Set `INPUT_DIR` in the next config cell to match your attached dataset's mount path.

**Long continuous recordings (few files, each spanning minutes, instead of many short clips):**
run Stage 2 with `--event-detect-labels` for any label whose long files have scattered
impact events (e.g. `hit`, `background_self_fire`) so windows land on each event instead
of arbitrary fixed slices, and make sure your `manifest.csv` has the `start_sec`/`duration_sec`
columns (current `make_features.py` writes these automatically). The split cell below
(`time_block_split`) then automatically falls back from whole-file holdout to a purged
within-file time-block split for any label with too few distinct source files -- see its
docstring for why.


In [ ]:
import csv, datetime, glob, json, os, random, re
from collections import defaultdict, Counter

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler

print("torch:", torch.__version__, "cuda available:", torch.cuda.is_available())


In [ ]:
# Kaggle mounts every attached dataset somewhere under /kaggle/input/,
# but the exact subpath (username, dataset slug, internal zip folder
# names) shifts across accounts and across re-uploads of the same
# dataset -- never hardcode past this root. find_manifest() below
# locates manifest.csv (and therefore the real processed-data root)
# dynamically, so nothing else needs adjusting when that path shifts.
KAGGLE_INPUT_ROOT = "/kaggle/input"
OUTPUT_DIR = "/kaggle/working"

# Raw labels recorded on the Pi that should collapse into a single training
# class. Add an entry here whenever you record a new hard-negative session
# under its own label (e.g. driving + self-fire noise) instead of directly
# under "background" -- this keeps the raw data traceable (you can still
# measure false-positive rate per source) while training stays a plain
# binary hit/background classifier.
LABEL_MAP = {
    "background_self_fire": "background",
}

# Labels recorded as continuous ambient sessions (dataset_collector's
# `--mode continuous`), where dataset_collector writes many small
# fixed-length chunk files back-to-back rather than one big file. Adjacent
# chunks from the same session are near-duplicate audio, so these labels
# need session-aware splitting below -- naive whole-file holdout would
# happily put chunk #47 in train and chunk #48 (five seconds later, same
# room, same motor noise) in validation.
#
# Labels NOT listed here are treated as independent discrete events
# (dataset_collector's `--mode trigger`, e.g. hit / background_self_fire).
# Even when many were captured in one quick sitting, each is a physically
# distinct strike/bang, so plain whole-file holdout is safe for them
# regardless of file count or how close together in time they were recorded.
CONTINUOUS_LABELS = {"background"}

# Files whose recording timestamps (parsed from the filename dataset_collector
# gives them) are within this many seconds of each other count as the same
# continuous recording session.
SESSION_GAP_SECONDS = 30.0

# Guard gap (seconds) dropped from both sides of the train/val cut when a
# continuous label ends up with only one session to split (see
# time_block_split's docstring) -- must comfortably exceed how long ambient
# noise stays audibly similar to itself, or a train window and a val window
# can still be near-duplicates despite formally being "different windows."
GUARD_SECONDS = 8.0

VAL_FRAC = 0.2
EPOCHS = 40
BATCH_SIZE = 16
LR = 3e-4  # lowered from 1e-3 -- Adam + BatchNorm + batch_size=16 + heavy class-imbalance oversampling was producing occasional loss spikes (up to 5x+) even with gradient clipping; a smaller LR shrinks every step regardless of Adam's adaptive scaling, which clipping alone doesn't fully control
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


In [ ]:
def find_manifest(search_root=KAGGLE_INPUT_ROOT):
    """Locate manifest.csv anywhere under search_root. Kaggle's mount path
    (username, dataset slug, and whatever internal folder names were inside
    your uploaded zip) is never stable across accounts or re-uploads, so it
    is never safe to hardcode past search_root -- this walks the whole tree
    instead."""
    candidates = sorted(glob.glob(os.path.join(search_root, "**", "manifest.csv"), recursive=True))

    if not candidates:
        raise FileNotFoundError(
            f"No manifest.csv found anywhere under {search_root}. Make sure your "
            "processed-dataset zip is attached to this notebook (right sidebar -> "
            "Add Input) and that it actually contains manifest.csv."
        )

    if len(candidates) == 1:
        print(f"Found manifest.csv: {candidates[0]}")
        return candidates[0]

    print(f"Found {len(candidates)} manifest.csv files under {search_root}:")
    for c in candidates:
        print(f"  - {c}  (modified {datetime.datetime.fromtimestamp(os.path.getmtime(c))})")
    chosen = max(candidates, key=os.path.getmtime)
    print(f"Multiple candidates found -- using the most recently modified: {chosen}")
    print("If that's the wrong one, pass search_root explicitly, or delete the stale input.")
    return chosen


def find_file(filename, search_root=KAGGLE_INPUT_ROOT):
    """Recursively locate filename anywhere under search_root. Used as a
    fallback when a path built from the manifest's own directory doesn't
    exist -- e.g. if a dataset re-upload nested the .npy files one level
    differently than the manifest.csv that references them."""
    return sorted(glob.glob(os.path.join(search_root, "**", filename), recursive=True))


def resolve_npy_path(processed_dir, relative_path):
    """The manifest's 'path' column is relative to wherever manifest.csv
    itself lives, so this should always resolve on the first try -- the
    recursive fallback only matters if a dataset re-upload/rename ever
    nests things differently than when the manifest was generated."""
    direct = os.path.join(processed_dir, relative_path)
    if os.path.exists(direct):
        return direct
    matches = find_file(os.path.basename(relative_path))
    if not matches:
        raise FileNotFoundError(
            f"Could not locate {relative_path} under {processed_dir} or anywhere "
            f"under {KAGGLE_INPUT_ROOT}."
        )
    return matches[0]


def load_manifest():
    manifest_path = find_manifest()
    processed_dir = os.path.dirname(manifest_path)
    with open(manifest_path, newline="") as f:
        rows = list(csv.DictReader(f))
    for r in rows:
        r["raw_label"] = r["label"]  # preserved for post-hoc analysis (e.g. FP source breakdown)
        r["label"] = LABEL_MAP.get(r["label"], r["label"])
    return rows, processed_dir


rows, INPUT_DIR = load_manifest()
labels = sorted(set(r["label"] for r in rows))
label_to_idx = {label: i for i, label in enumerate(labels)}
print("labels:", label_to_idx)
print("total windows:", len(rows))
for label in labels:
    print(" ", label, sum(1 for r in rows if r["label"] == label))


In [ ]:
FILENAME_TS_RE = re.compile(r"_(\d{8})_(\d{6})_(\d+)_\d+\.wav$")


def parse_timestamp(source_wav):
    """dataset_collector names files <label>_<YYYYMMDD>_<HHMMSS>_<frac>_<idx>.wav
    -- pull the wall-clock moment the file was written out of that."""
    m = FILENAME_TS_RE.search(source_wav)
    date, time_, frac = m.group(1), m.group(2), m.group(3)
    ts = datetime.datetime.strptime(date + time_, "%Y%m%d%H%M%S")
    return ts + datetime.timedelta(microseconds=int(frac.ljust(6, "0")[:6]))


def build_sessions(file_rows_by_file, gap_seconds):
    """file_rows_by_file: {source_wav: [manifest rows]}. Returns a list of
    sessions, each a list of source_wav names in chronological order, where
    consecutive files less than gap_seconds apart (end of one to start of
    the next) are considered the same continuous recording session."""
    file_span = {}
    for source_wav, rows in file_rows_by_file.items():
        ts = parse_timestamp(source_wav)
        duration = max(float(r["start_sec"]) + float(r["duration_sec"]) for r in rows)
        file_span[source_wav] = (ts, duration)

    ordered = sorted(file_span, key=lambda f: file_span[f][0])
    sessions = []
    current = []
    prev_end = None
    for f in ordered:
        ts, duration = file_span[f]
        if prev_end is not None and (ts - prev_end).total_seconds() > gap_seconds:
            sessions.append(current)
            current = []
        current.append(f)
        prev_end = ts + datetime.timedelta(seconds=duration)
    if current:
        sessions.append(current)
    return sessions, file_span


def time_block_split(rows, val_frac, seed, continuous_labels=CONTINUOUS_LABELS,
                      gap_seconds=SESSION_GAP_SECONDS, guard_seconds=GUARD_SECONDS):
    """Two strategies, chosen per label:

    - Not in continuous_labels (discrete triggered events, e.g. hit,
      background_self_fire): plain whole-file holdout. Each file is one
      physically independent strike/event, so file count and recording
      cadence don't matter for leakage -- holding out whole files is safe.

    - In continuous_labels (ambient sessions, e.g. background, which
      dataset_collector may have written as many small fixed-length chunks
      back-to-back): files are first grouped into *sessions* using their
      recording timestamps (a run of files less than gap_seconds apart is
      one session). With >=2 sessions, whole sessions are held out for
      validation -- clean, no leakage, since separate sessions don't share
      adjacency. With exactly one session, there's nothing to hold out
      wholesale, so instead the session's own concatenated timeline is
      split (last val_frac -> validation) with a guard_seconds purge band
      around the cut, dropped from both sides, so no train window and no
      val window are close enough in time to be near-duplicate audio.
    """
    rows_by_label = defaultdict(list)
    for r in rows:
        rows_by_label[r["label"]].append(r)

    rng = random.Random(seed)
    train_rows, val_rows = [], []

    for label, label_rows in rows_by_label.items():
        if label not in continuous_labels:
            files = sorted(set(r["source_wav"] for r in label_rows))
            rng.shuffle(files)
            n_val = max(1, round(len(files) * val_frac))
            val_files = set(files[:n_val])
            for r in label_rows:
                (val_rows if r["source_wav"] in val_files else train_rows).append(r)
            continue

        by_file = defaultdict(list)
        for r in label_rows:
            by_file[r["source_wav"]].append(r)
        sessions, file_span = build_sessions(by_file, gap_seconds)

        if len(sessions) >= 2:
            session_sizes = [sum(len(by_file[f]) for f in s) for s in sessions]
            order = list(range(len(sessions)))
            rng.shuffle(order)
            target_val = round(sum(session_sizes) * val_frac)
            val_session_idx, running = set(), 0
            for i in order:
                if running >= target_val:
                    break
                val_session_idx.add(i)
                running += session_sizes[i]
            for i, session in enumerate(sessions):
                dest = val_rows if i in val_session_idx else train_rows
                for f in session:
                    dest.extend(by_file[f])
        else:
            session = sessions[0]
            session_start = min(file_span[f][0] for f in session)
            session_end = max((file_span[f][0] + datetime.timedelta(seconds=file_span[f][1]) - session_start).total_seconds() for f in session)
            split_point = session_end * (1 - val_frac)

            for f in session:
                offset = (file_span[f][0] - session_start).total_seconds()
                for r in by_file[f]:
                    start = offset + float(r["start_sec"])
                    end = start + float(r["duration_sec"])
                    if end <= split_point - guard_seconds:
                        train_rows.append(r)
                    elif start >= split_point + guard_seconds:
                        val_rows.append(r)
                    # else: inside the guard band around the cut -- dropped from both sets

    return train_rows, val_rows


train_rows, val_rows = time_block_split(rows, VAL_FRAC, SEED)
print(f"train windows: {len(train_rows)}  val windows: {len(val_rows)}")

for label in labels:
    label_rows = [r for r in rows if r["label"] == label]
    n_files = len(set(r["source_wav"] for r in label_rows))
    if label in CONTINUOUS_LABELS:
        sessions, _ = build_sessions(
            {r["source_wav"]: [rr for rr in label_rows if rr["source_wav"] == r["source_wav"]] for r in label_rows},
            SESSION_GAP_SECONDS,
        )
        print(f"  {label}: {n_files} file(s) -> {len(sessions)} session(s) -> "
              f"{'whole-session holdout' if len(sessions) >= 2 else 'single-session purged time-block split'}")
    else:
        print(f"  {label}: {n_files} file(s) -> whole-file holdout (discrete events)")


In [ ]:
def compute_stats(rows, processed_dir):
    vals = [np.load(resolve_npy_path(processed_dir, r["path"])) for r in rows]
    arr = np.stack(vals)
    return float(arr.mean()), float(arr.std())

mean, std = compute_stats(train_rows, INPUT_DIR)
print(f"train feature mean={mean:.3f} std={std:.3f}")

def spec_augment(feat, freq_mask=8, time_mask=12):
    feat = feat.copy()
    n_mels, n_frames = feat.shape
    f0 = random.randint(0, max(0, n_mels - freq_mask))
    feat[f0:f0 + freq_mask, :] = 0.0
    t0 = random.randint(0, max(0, n_frames - time_mask))
    feat[:, t0:t0 + time_mask] = 0.0
    return feat

class SpecDataset(Dataset):
    def __init__(self, rows, processed_dir, label_to_idx, mean, std, augment):
        self.rows = rows
        self.processed_dir = processed_dir
        self.label_to_idx = label_to_idx
        self.mean = mean
        self.std = std
        self.augment = augment

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows[idx]
        feat = np.load(resolve_npy_path(self.processed_dir, row["path"]))
        feat = (feat - self.mean) / self.std
        if self.augment:
            feat = spec_augment(feat)
        x = torch.from_numpy(feat).float().unsqueeze(0)
        y = self.label_to_idx[row["label"]]
        return x, y

train_ds = SpecDataset(train_rows, INPUT_DIR, label_to_idx, mean, std, augment=True)
val_ds = SpecDataset(val_rows, INPUT_DIR, label_to_idx, mean, std, augment=False)

class_counts = defaultdict(int)
for r in train_rows:
    class_counts[r["label"]] += 1
sample_weights = [1.0 / class_counts[r["label"]] for r in train_rows]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(train_rows), replacement=True)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

sample_x, _ = train_ds[0]
print("model input shape:", tuple(sample_x.shape))


In [ ]:
class ImpactCNN(nn.Module):
    def __init__(self, n_classes=2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Linear(64, n_classes)

    def forward(self, x):
        x = self.features(x)
        x = x.flatten(1)
        return self.classifier(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ImpactCNN(n_classes=len(labels)).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"device: {device}, params: {n_params:,}")


In [ ]:
def evaluate(model, loader, device, criterion=None):
    model.eval()
    tp = tn = fp = fn = 0
    total_loss, n = 0.0, 0
    hit_idx = label_to_idx.get("hit", 1)
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            if criterion is not None:
                total_loss += criterion(logits, y).item() * x.size(0)
                n += x.size(0)
            pred = logits.argmax(1)
            for p, t in zip(pred.tolist(), y.tolist()):
                tp += p == hit_idx and t == hit_idx
                tn += p != hit_idx and t != hit_idx
                fp += p == hit_idx and t != hit_idx
                fn += p != hit_idx and t == hit_idx
    total = tp + tn + fp + fn
    accuracy = (tp + tn) / total if total else 0.0
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    return {"accuracy": accuracy, "precision": precision, "recall": recall, "f1": f1,
            "tp": tp, "tn": tn, "fp": fp, "fn": fn,
            "loss": (total_loss / n) if n else None}


def collect_predictions(model, loader, device):
    model.eval()
    preds = []
    with torch.no_grad():
        for x, _ in loader:
            preds.extend(model(x.to(device)).argmax(1).cpu().tolist())
    return preds


In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss()

# Non-augmented, non-resampled view of the training set, used only to measure
# a clean train loss/F1 each epoch -- comparable to val instead of muddied by
# the SGD minibatches' augmentation and class-balanced resampling.
train_eval_ds = SpecDataset(train_rows, INPUT_DIR, label_to_idx, mean, std, augment=False)
train_eval_loader = DataLoader(train_eval_ds, batch_size=BATCH_SIZE, shuffle=False)

os.makedirs(OUTPUT_DIR, exist_ok=True)
best_f1 = -1.0
best_state = None
history = {"train_loss": [], "val_loss": [], "train_f1": [], "val_f1": []}

for epoch in range(1, EPOCHS + 1):
    model.train()
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        # Clips exploding gradients before they hit the optimizer step.
        # With a tiny, heavily class-imbalanced dataset (WeightedRandomSampler
        # oversamples ~70-80 hit examples against ~900 background) plus the
        # per-batch noise/time-shift augmentation, an occasional bad batch can
        # produce a large gradient that Adam turns into a destructive step --
        # seen as loss briefly spiking 5-10x and the model collapsing toward
        # predicting almost everything as "hit" for a few epochs before
        # recovering. Clipping caps the step size regardless of how extreme
        # any single batch's gradient gets.
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

    train_metrics = evaluate(model, train_eval_loader, device, criterion)
    val_metrics = evaluate(model, val_loader, device, criterion)
    history["train_loss"].append(train_metrics["loss"])
    history["val_loss"].append(val_metrics["loss"])
    history["train_f1"].append(train_metrics["f1"])
    history["val_f1"].append(val_metrics["f1"])

    print(
        f"epoch {epoch:3d}  train_loss={train_metrics['loss']:.4f}  val_loss={val_metrics['loss']:.4f}  "
        f"train_f1={train_metrics['f1']:.3f}  val_f1={val_metrics['f1']:.3f}  "
        f"(tp={val_metrics['tp']} fp={val_metrics['fp']} fn={val_metrics['fn']} tn={val_metrics['tn']})"
    )

    if val_metrics["f1"] >= best_f1:
        best_f1 = val_metrics["f1"]
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

model.load_state_dict(best_state)
print(f"best val f1={best_f1:.3f}")


In [ ]:
idx_to_label = {i: label for label, i in label_to_idx.items()}
hit_idx = label_to_idx.get("hit", 1)

val_preds = collect_predictions(model, val_loader, device)
val_targets = [label_to_idx[r["label"]] for r in val_rows]

val_raw_counts = Counter(r.get("raw_label", r["label"]) for r in val_rows)
train_raw_counts = Counter(r.get("raw_label", r["label"]) for r in train_rows)
print("validation set composition by original recording source:")
for src_label, n in val_raw_counts.items():
    print(f"  {src_label}: {n} windows (train has {train_raw_counts.get(src_label, 0)})")
    if train_raw_counts.get(src_label, 0) == 0 and val_raw_counts.get(src_label, 0) > 0:
        print(f"    NOTE: {src_label} has zero training windows -- it's entirely held out this run.")

n_classes = len(label_to_idx)
cm = np.zeros((n_classes, n_classes), dtype=int)
for t, p in zip(val_targets, val_preds):
    cm[t, p] += 1

fp_by_source = Counter()
for r, t, p in zip(val_rows, val_targets, val_preds):
    if p == hit_idx and t != hit_idx:
        fp_by_source[r.get("raw_label", r["label"])] += 1


def plot_training_curves(history, out_dir):
    epochs = range(1, len(history["train_loss"]) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

    ax = axes[0]
    ax.plot(epochs, history["train_loss"], label="Train loss", color="#2b6e76", linewidth=1.8)
    ax.plot(epochs, history["val_loss"], label="Val loss", color="#c8862a", linewidth=1.8)
    ax.set_title("Loss per epoch")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Cross-entropy loss")
    ax.legend(frameon=False)
    ax.grid(alpha=0.25)

    ax = axes[1]
    ax.plot(epochs, history["train_f1"], label="Train F1 (hit)", color="#2b6e76", linewidth=1.8)
    ax.plot(epochs, history["val_f1"], label="Val F1 (hit)", color="#c8862a", linewidth=1.8)
    ax.set_title("F1-score (hit class) per epoch")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("F1-score")
    ax.set_ylim(0, 1.02)
    ax.legend(frameon=False)
    ax.grid(alpha=0.25)

    fig.tight_layout()
    fig.savefig(os.path.join(out_dir, "training_curves.png"), dpi=150)
    plt.show()


def plot_confusion_matrix(cm, class_names, out_dir):
    fig, ax = plt.subplots(figsize=(4.6, 4.2))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(len(class_names)))
    ax.set_yticks(range(len(class_names)))
    ax.set_xticklabels(class_names)
    ax.set_yticklabels(class_names)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_title("Validation confusion matrix (best epoch)")

    thresh = cm.max() / 2 if cm.max() else 0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=13, fontweight="bold",
                    color="white" if cm[i, j] > thresh else "black")

    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    fig.savefig(os.path.join(out_dir, "confusion_matrix.png"), dpi=150)
    plt.show()


def plot_fp_breakdown(fp_by_source, out_dir):
    if not fp_by_source:
        print("No false positives in validation -- nothing to break down.")
        return
    names = list(fp_by_source.keys())
    counts = [fp_by_source[n] for n in names]
    fig, ax = plt.subplots(figsize=(5.5, 0.6 * len(names) + 1))
    ax.barh(names, counts, color="#c8862a")
    for i, c in enumerate(counts):
        ax.text(c, i, f" {c}", va="center")
    ax.set_xlabel("False positives (predicted hit)")
    ax.set_title("Validation false positives by original recording source")
    fig.tight_layout()
    fig.savefig(os.path.join(out_dir, "fp_breakdown.png"), dpi=150)
    plt.show()


plot_training_curves(history, OUTPUT_DIR)
plot_confusion_matrix(cm, [idx_to_label[i] for i in range(n_classes)], OUTPUT_DIR)
plot_fp_breakdown(fp_by_source, OUTPUT_DIR)


In [ ]:
model.eval().cpu()
sample_shape = tuple(sample_x.shape)  # (1, n_mels, n_frames)
dummy_input = torch.randn(1, *sample_shape)

onnx_path = os.path.join(OUTPUT_DIR, "impact_cnn.onnx")
torch.onnx.export(
    model,
    dummy_input,
    onnx_path,
    input_names=["spectrogram"],
    output_names=["logits"],
    opset_version=12,
    dynamic_axes=None,  # fixed shape on purpose -- must match Stage 4's window length exactly
    dynamo=False,  # force the legacy exporter -- the newer dynamo path needs the onnxscript package, not installed by default
)
print("exported", onnx_path)

with open(os.path.join(OUTPUT_DIR, "labels.json"), "w") as f:
    json.dump(label_to_idx, f, indent=2)
with open(os.path.join(OUTPUT_DIR, "feature_stats.json"), "w") as f:
    json.dump({"mean": mean, "std": std, "input_shape": list(sample_shape)}, f, indent=2)

print("labels.json and feature_stats.json written to", OUTPUT_DIR)


In [ ]:
!pip install -q onnxruntime

import onnxruntime as ort

sess = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
with torch.no_grad():
    torch_out = model(dummy_input).numpy()
onnx_out = sess.run(None, {"spectrogram": dummy_input.numpy()})[0]

max_diff = np.abs(torch_out - onnx_out).max()
print("max abs diff torch vs onnx:", max_diff)
assert max_diff < 1e-4, "ONNX export mismatch!"
print("ONNX export verified.")


## Done

Download these three files from the notebook's **Output** panel (`/kaggle/working`) and copy them
to `ai_audio/model/` on the Pi for Stage 4:

- `impact_cnn.onnx`
- `labels.json`
- `feature_stats.json`
